In [1]:
# Kruskal MST
import numpy as np

class DisjointSet:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    
    def find(self, u):
        if self.parent[u] != u:
            self.parent[u] = self.find(self.parent[u])
        return self.parent[u]
    
    def union(self, u, v):
        root_u = self.find(u)
        root_v = self.find(v)
        if root_u != root_v:
            if self.rank[root_u] > self.rank[root_v]:
                self.parent[root_v] = root_u
            elif self.rank[root_u] < self.rank[root_v]:
                self.parent[root_u] = root_v
            else:
                self.parent[root_v] = root_u
                self.rank[root_u] += 1
            return True
        return False

def kruskal_mst(X):
    n = len(X)
    edges = []
    for i in range(n):
        for j in range(i+1, n):
            dist = np.linalg.norm(X[i] - X[j])
            edges.append((dist, i, j))
    edges.sort(key=lambda x: x[0])

    ds = DisjointSet(n)
    mst = []
    for dist, u, v in edges:
        if ds.union(u, v):
            mst.append((u, v, dist))
        if len(mst) == n-1:
            break
    return mst

def mst_clustering_kruskal(X, k):
    mst = kruskal_mst(X)
    mst_sorted = sorted(mst, key=lambda x: x[2], reverse=True)
    edges_to_remove = mst_sorted[:k-1]
    edges_to_keep = mst_sorted[k-1:]

    n = len(X)
    adj = [[] for _ in range(n)]
    for u, v, dist in edges_to_keep:
        adj[u].append(v)
        adj[v].append(u)

    visited = [False]*n
    clusters = []

    def dfs(node, cluster):
        visited[node] = True
        cluster.append(node)
        for nei in adj[node]:
            if not visited[nei]:
                dfs(nei, cluster)

    for i in range(n):
        if not visited[i]:
            cluster = []
            dfs(i, cluster)
            clusters.append(cluster)
    return clusters

X = np.array([[1, 2], [2, 3], [10, 10], [11, 11]])
clusters = mst_clustering_kruskal(X, k=2)
print("Kruskal MST clusters:", clusters)


Kruskal MST clusters: [[0, 1], [2, 3]]


In [2]:
# Prims MST
import numpy as np
import heapq

def prim_mst(X):
    n = len(X)
    visited = [False]*n
    min_heap = [(0, 0, -1)]
    mst = []

    while min_heap and len(mst) < n-1:
        w, u, p = heapq.heappop(min_heap)
        if visited[u]:
            continue
        visited[u] = True
        if p != -1:
            mst.append((p, u, w))
        for v in range(n):
            if not visited[v]:
                dist = np.linalg.norm(X[u] - X[v])
                heapq.heappush(min_heap, (dist, v, u))
    return mst

def mst_clustering_prim(X, k):
    mst = prim_mst(X)
    mst_sorted = sorted(mst, key=lambda x: x[2], reverse=True)
    edges_to_remove = mst_sorted[:k-1]
    edges_to_keep = mst_sorted[k-1:]

    n = len(X)
    adj = [[] for _ in range(n)]
    for u, v, dist in edges_to_keep:
        adj[u].append(v)
        adj[v].append(u)

    visited = [False]*n
    clusters = []

    def dfs(node, cluster):
        visited[node] = True
        cluster.append(node)
        for nei in adj[node]:
            if not visited[nei]:
                dfs(nei, cluster)

    for i in range(n):
        if not visited[i]:
            cluster = []
            dfs(i, cluster)
            clusters.append(cluster)
    return clusters

clusters = mst_clustering_prim(X, k=2)
print("Prim MST clusters:", clusters)


Prim MST clusters: [[0, 1], [2, 3]]


In [3]:
# BIRCH
import numpy as np

class CFNode:
    def __init__(self, max_entries=3, is_leaf=True):
        self.max_entries = max_entries
        self.is_leaf = is_leaf
        self.entries = []
        self.parent = None
    
    def add_entry(self, entry):
        self.entries.append(entry)
        entry.parent = self

class CFEntry:
    def __init__(self, N, LS, SS, child=None):
        self.N = N
        self.LS = LS
        self.SS = SS
        self.child = child
        self.parent = None

    def centroid(self):
        return self.LS / self.N

    def radius(self):
        return np.sqrt(np.sum(self.SS / self.N - (self.centroid() ** 2)))

    def absorb(self, point):
        self.N += 1
        self.LS += point
        self.SS += point ** 2

class BIRCH:
    def __init__(self, threshold=1.5, max_entries=3):
        self.threshold = threshold
        self.max_entries = max_entries
        self.root = CFNode(max_entries=self.max_entries, is_leaf=True)
    
    def insert(self, point, node=None):
        if node is None:
            node = self.root
        
        if node.is_leaf:
            closest_entry = None
            min_dist = float('inf')
            for entry in node.entries:
                dist = np.linalg.norm(entry.centroid() - point)
                if dist < min_dist:
                    min_dist = dist
                    closest_entry = entry
            
            if closest_entry and closest_entry.radius() < self.threshold:
                closest_entry.absorb(point)
            else:
                new_entry = CFEntry(1, point.copy(), point**2)
                node.add_entry(new_entry)
            
            if len(node.entries) > node.max_entries:
                self.split(node)
        else:
            closest_entry = None
            min_dist = float('inf')
            for entry in node.entries:
                dist = np.linalg.norm(entry.centroid() - point)
                if dist < min_dist:
                    min_dist = dist
                    closest_entry = entry
            self.insert(point, closest_entry.child)
    
    def split(self, node):
        entries = node.entries
        node.entries = []

        max_dist = 0
        seed1, seed2 = None, None
        for i in range(len(entries)):
            for j in range(i+1, len(entries)):
                dist = np.linalg.norm(entries[i].centroid() - entries[j].centroid())
                if dist > max_dist:
                    max_dist = dist
                    seed1, seed2 = entries[i], entries[j]

        node1 = CFNode(max_entries=node.max_entries, is_leaf=node.is_leaf)
        node2 = CFNode(max_entries=node.max_entries, is_leaf=node.is_leaf)

        node1.add_entry(seed1)
        node2.add_entry(seed2)

        for e in entries:
            if e == seed1 or e == seed2:
                continue
            dist1 = np.linalg.norm(e.centroid() - seed1.centroid())
            dist2 = np.linalg.norm(e.centroid() - seed2.centroid())
            if dist1 < dist2:
                node1.add_entry(e)
            else:
                node2.add_entry(e)

        if node.parent is None:
            new_root = CFNode(max_entries=node.max_entries, is_leaf=False)
            new_root.add_entry(CFEntry(self._sum_N(node1), self._sum_LS(node1), self._sum_SS(node1), node1))
            new_root.add_entry(CFEntry(self._sum_N(node2), self._sum_LS(node2), self._sum_SS(node2), node2))
            self.root = new_root
            node1.parent = self.root
            node2.parent = self.root
        else:
            parent = node.parent
            parent.entries.remove(next(e for e in parent.entries if e.child == node))
            parent.add_entry(CFEntry(self._sum_N(node1), self._sum_LS(node1), self._sum_SS(node1), node1))
            parent.add_entry(CFEntry(self._sum_N(node2), self._sum_LS(node2), self._sum_SS(node2), node2))
            if len(parent.entries) > parent.max_entries:
                self.split(parent)

    def _sum_N(self, node):
        return sum(e.N for e in node.entries)

    def _sum_LS(self, node):
        return sum(e.LS for e in node.entries)

    def _sum_SS(self, node):
        return sum(e.SS for e in node.entries)

    def print_tree(self, node=None, level=0):
        if node is None:
            node = self.root
        print("  " * level + f"Node (leaf={node.is_leaf}) with {len(node.entries)} entries")
        for e in node.entries:
            print("  " * (level+1) + f"Entry: N={e.N}, centroid={e.centroid()}, radius={e.radius()}")
            if e.child:
                self.print_tree(e.child, level+2)

birch = BIRCH(threshold=1.5, max_entries=3)
points = np.array([[1, 2], [2, 3], [1, 1], [10, 10], [11, 11], [10, 9]])

for p in points:
    birch.insert(p)

birch.print_tree()


Node (leaf=True) with 2 entries
  Entry: N=4, centroid=[3.5 4. ], radius=5.172040216394301
  Entry: N=2, centroid=[10.5 10. ], radius=1.118033988749895
